# Shadow-1-Whisper-0.8B — Fine-Tune & Export (free Colab T4)

Fine-tunes **`unsloth/Qwen3.5-0.8B`** on ShadowTalk's SFT dataset (`shadowtalk_sft.jsonl`, 620
examples: 45% shadowvisual JSON / 25% agentic missions / 20% grounded RAG / 10% identity+multilingual)
and exports **three artifacts**:

| Artifact | Repo | Consumer |
|---|---|---|
| Merged 16-bit weights | `shadowtalkai/Shadow-1-Whisper-0.8B` | base for GGUF + ONNX transplant |
| GGUF `q4_k_m` | `shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` | Tauri Desktop (llama.cpp) |
| ONNX q4 (WebGPU) | `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX` | Browser worker (`@huggingface/transformers` v4+) |

**Before running:** get a Hugging Face **write** token (fine-grained, repos: *write*), create the three
repos under `shadowtalkai/` (or let Cell 4 create them), and upload `shadowtalk_sft.jsonl` next to this
notebook. Everything below runs on a free T4 (~3 GB VRAM for LoRA, fp16).

## Cell 1 — Setup
Fails fast if the runtime has no GPU, then installs the stack **exactly as the official
unsloth Qwen3.5 Vision Colab notebook does** — pinned `torch 2.8.0`, `transformers==5.2.0`,
`trl==0.22.2`, `tokenizers≤0.23.0`, xformers + `unsloth install-kernels` — plus the
export-side ONNX tools. Takes ~3-5 min on a fresh Colab runtime.

In [ ]:
# 1 ─ Setup — fail-fast GPU check, then the PINNED install stack
#
# Check the GPU BEFORE installing anything: on a CPU runtime this dies in 2s
# instead of after 3-5 min of pip. torch ships preinstalled on Colab.
# And refuse to reinstall into a session that already imported the stack:
# pip/uv replacing binaries under a live interpreter is the #1 cause of the
# 'RuntimeError: operator torchvision::nms does not exist' import crash — the
# old library stays registered in-process and collides with the new one.
# Only a runtime restart clears that; reinstalling cannot.
import sys
_already = [m for m in ("unsloth", "unsloth_zoo", "torchvision", "transformers", "trl", "peft") if m in sys.modules]
assert not _already, (
    f"{', '.join(_already)} already imported in this session — re-running the installer now would "
    "corrupt the loaded binary state. Runtime → Restart session, then re-run this cell."
)
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  capability {torch.cuda.get_device_capability(0)}")

# Install the EXACT combo the official unsloth Qwen3.5 Vision Colab notebook
# (unslothai/notebooks · nb/Qwen3_5_(0_8B)_Vision.ipynb — the reference notebook
# for our base model) installs. The pins are load-bearing: PyPI's latest
# transformers is 5.24.x, but unsloth's Qwen3.5 kernels are built and tested
# against transformers==5.2.0 + trl==0.22.2 + tokenizers<=0.23.0 — a loose
# 'transformers>=5.2.0' resolves to an untested pairing and breaks the patched
# forward pass / SFTConfig API.
!pip install --upgrade -qqq uv
import importlib.util, os
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

# torchvision hygiene — the classic unsloth-on-Colab import crash is a
# torchvision whose version metadata pairs with torch but whose compiled ops
# fail to load ('operator torchvision::nms does not exist'). Re-unpack the
# exact wheel resolved above: --no-deps (never touch torch), --no-cache-dir
# (never reuse a possibly-stale unpack), --force-reinstall (clean uninstall
# first). unsloth's own compatibility check prescribes exactly this
# remediation if its probe ever fails; doing it here makes that unreachable.
import importlib.metadata as _im
_tv = _im.version("torchvision")
!pip install --force-reinstall --no-deps --no-cache-dir "torchvision=={_tv}"
print(f"torchvision {_tv}: clean re-unpack done (compiled-ops guard)")

# Export-side tools for Cell 4/5 (ONNX graph verify + tokenizer export). Deliberately
# WITHOUT -U so nothing drags huggingface_hub/safetensors away from what
# transformers==5.2.0 pins.
!pip install -q onnx onnxruntime onnx_ir sentencepiece protobuf  # onnx_ir: MatMulNBits quantizer dep (ORT >=1.20)

# Verify the pinned combo before burning GPU time. If an assert ever fires
# mid-session: Runtime → Restart session, then re-run this cell.
import importlib.metadata as im
for pkg in ["unsloth", "unsloth_zoo", "transformers", "tokenizers", "trl", "peft",
            "bitsandbytes", "onnx", "onnxruntime", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"{pkg:18s} MISSING")
assert im.version("transformers") == "5.2.0", "transformers must be exactly 5.2.0 (unsloth Qwen3.5 tested pin)"
assert im.version("trl") == "0.22.2", "trl must be exactly 0.22.2 (official Qwen3.5 Vision notebook pin)"

# T4 note: Turing (sm_75) has no native bf16 — 'False' here is EXPECTED, not an
# error. Cell 3 auto-selects fp16 on T4 (bf16 only on Ampere+/L4).
print("bf16 supported:", torch.cuda.is_bf16_supported())


## Cell 2 — Auth & Model Load
Prompts for the HF token securely (`getpass`, never printed), loads the base model in **bf16/fp16 at full
precision** (no 4-bit: a 0.8B model + LoRA fits comfortably in ~3 GB on the T4), and attaches LoRA adapters
to the **language layers only** — the vision stack is frozen because Shadow-1-Whisper is a text-contract model.

In [ ]:
# 2 ─ Auth & model load (secure token + LoRA on language layers)
#
# Import unsloth FIRST — before any user interaction. If the environment is
# broken (kernel / torchvision mismatch), the crash must surface here, not
# after the user has already been prompted for a token.
from unsloth import FastVisionModel
import getpass
from huggingface_hub import login

HF_TOKEN = getpass.getpass("🔑 Paste your Hugging Face WRITE token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "That doesn't look like an HF token (should start with 'hf_')"
login(token=HF_TOKEN)

BASE_MODEL = "unsloth/Qwen3.5-0.8B"
SEED = 3407

# load_in_4bit=False → full-precision LoRA: ~3 GB VRAM on the T4 (fp16; bf16 on Ampere+)
model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

# Attach LoRA — language layers only, vision frozen (text-contract model)
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
)

FastVisionModel.for_training(model)
model.print_trainable_parameters()

tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
print("Base model loaded:", BASE_MODEL)
print("dtype:", next(model.parameters()).dtype)

## Cell 3 — Train (SFT on the chat-template)
Loads `shadowtalk_sft.jsonl` (upload it next to the notebook — the cell prompts if missing), renders every
record through `tokenizer.apply_chat_template(..., add_generation_prompt=False)` and trains with
`SFTTrainer` for **3 epochs @ lr 2e-4**, seq len 2048. On a free T4 expect **~25-40 min** for the 620-example run.

In [ ]:
# 3 ─ Train — chat-templated SFT

# Loss-surface note: SFT loss covers all assistant turns, so tool-card turns inside
# multi-turn records must never be reachable as model output. Every card-bearing record
# in shadowtalk_sft.jsonl opens with a request the runtime tool detector
# (useToolOrchestrator.detectToolSync) intercepts before the model is called — calc/
# calculate prefix, bare arithmetic, 'search the web for …', or an explicit /image
# command. The only exceptions are database_query / file_manager, which have no
# auto-detection: their records intentionally teach the honest 'not yet deployed'
# runtime reply instead of a fabricated tool success.
import json, os
from pathlib import Path
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

DATA_FILE = "shadowtalk_sft.jsonl"
if not Path(DATA_FILE).exists():
    print("Upload shadowtalk_sft.jsonl now (from training/shadow-1-whisper/)…")
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]

rows = [json.loads(line) for line in open(DATA_FILE, encoding="utf-8")]
assert rows and all("messages" in r for r in rows), "dataset rows must be {'messages': [...]}"
print(f"loaded {len(rows)} examples")

# Render each record with the model's own chat template.
# add_generation_prompt=False → the assistant turn is TRAINING TARGET, not a prompt prefix.
def render(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list([render(r) for r in rows]).shuffle(seed=SEED)
print("sample:\n" + dataset[0]["text"][:600] + "\n…")

MAX_SEQ = 2048
cfg = SFTConfig(
    output_dir="shadow-1-ckpt",
    dataset_text_field="text",
    max_length=MAX_SEQ,                     # trl 0.22.2 renamed max_seq_length → max_length
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch 16
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=(not torch.cuda.is_bf16_supported()),  # T4 = fp16; A100/L4 = bf16
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()

gpu_s = round(trainer_stats.metrics.get("train_runtime", 0), 1)
print(f"trained in {gpu_s}s  |  peak mem ≈ {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

## Cell 4 — Push Merged 16-bit + GGUF q4_k_m
Merges the LoRA adapters back into the base weights, pushes the merged model to
`shadowtalkai/Shadow-1-Whisper-0.8B`, then quantizes to **GGUF `q4_k_m`** and pushes
`shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` (desktop/Tauri runtime). GGUF quantization runs ~8-15 min on the T4.
Both repos are created **private** — flip them public after you've eyeballed them (see README).

In [ ]:
# 4 ─ Push merged 16-bit + GGUF q4_k_m
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

MERGED_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B"
GGUF_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-GGUF"

for repo in (MERGED_REPO, GGUF_REPO):
    api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True)

# 4a — merged 16-bit (the canonical weights + tokenizer + chat template)
model.push_to_hub_merged(
    MERGED_REPO,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print(f"✅ merged 16-bit → https://huggingface.co/{MERGED_REPO}")

# 4b — GGUF q4_k_m for the Tauri desktop runtime (llama.cpp)
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)
print(f"✅ GGUF q4_k_m → https://huggingface.co/{GGUF_REPO}")

## Cell 5 — Export the WebGPU ONNX package
`Qwen3.5-0.8B` is a **hybrid Gated DeltaNet + vision** architecture — we do NOT re-export the graph from
torch (fragile, slow, and would lose the battle-tested WebGPU kernels). Instead we **transplant**, with
the mapping verified offline against the real reference graph (397 initializers) and the real base
state dict (488 tensors):

1. download the reference WebGPU ONNX graph from `onnx-community/Qwen3.5-0.8B-ONNX`,
2. overwrite the decoder graphs' float initializers (`decoder_model_merged.onnx` + `_fp16`) with the
   fine-tuned merged weights — `graph_to_hf()` maps optimum's fused names
   (`model.layers.N.attn.q_proj.MatMul.weight`) to transformers-v5 names
   (`model.language_model.layers.N.self_attn.q_proj.weight`), `gdn→linear_attn`, the tied
   `lm_head→embed_tokens`, `q/k_norm.layernorm→q/k_norm`, and the layer-24 final norm →
   `model.language_model.norm`,
3. **transpose, never reshape** — the exporter folded the Linear transpose, so graph dims are the
   REVERSE of the HF `(out, in)` shapes; a same-size reshape would silently scramble every weight.
   The cell accepts only exact matches, true 2-D reversals, or a lossless axis-1 squeeze (conv1d),
   and fails loudly on anything else,
4. skip exporter constants and derived tensors (`A_neg_exp = exp(-A_log)`, `model.inv_freq`), and keep
   the reference's `embed_tokens*` / `vision_encoder*` graphs (LoRA never touched them — base-exact),
5. **regenerate the quantized decoder variants** (`_q4`, `_q4f16`) from the transplanted graphs with
   onnxruntime's MatMulNBits quantizer — the reference's quantized files carry BASE weights and would
   silently serve the base model to WebGPU users; the stale int8 `_quantized` variant is removed,
6. swap in the fine-tuned tokenizer + `chat_template` + `generation_config`,
7. verify every shipped decoder graph with **onnxruntime** (loads + documented IO), and
8. upload to `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX`.

The transplant reports matched/unmatched initializer counts and **fails loudly** below a 70% coverage
threshold, on any missing tensor, or on any size-ambiguous shape mismatch — a silently half-transplanted
model is worse than a failed export.

In [ ]:
# 5 ─ ONNX transplant: fine-tuned language weights → reference WebGPU graph
import glob, json, re, shutil
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, numpy_helper
from huggingface_hub import HfApi, snapshot_download

REF_REPO = "onnx-community/Qwen3.5-0.8B-ONNX"
MERGED_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B"   # same constant as cell 4a
MERGED_DIR = "Shadow-1-Whisper-0.8B-merged"           # local copy of cell-4a output
ONNX_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-ONNX"
api = HfApi(token=HF_TOKEN)  # defensive: cell 5 must not depend on cell 4's locals

if not Path(MERGED_DIR).exists():
    # cell 4 pushed to the Hub; pull the same merged weights back down
    MERGED_DIR = str(snapshot_download(MERGED_REPO, token=HF_TOKEN,
                                       local_dir="Shadow-1-Whisper-0.8B-merged"))

# 5.1 — reference graph → working dir
work = Path("Shadow-1-Whisper-0.8B-ONNX-work")
shutil.rmtree(work, ignore_errors=True)
ref_dir = snapshot_download(REF_REPO)
shutil.copytree(ref_dir, work, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
print("reference graph files:", len(list(work.rglob("*.onnx"))))

# 5.2 — merged state dict (safetensors shards)
from safetensors.torch import load_file
state = {}
for shard in sorted(glob.glob(f"{MERGED_DIR}/*.safetensors")):
    state.update(load_file(shard))
print(f"merged state dict: {len(state)} tensors")

# The transplant rules below were verified offline against the REAL reference graph
# (decoder_model_merged_fp16.onnx: 397 initializers) and the REAL base state dict
# (488 tensors). Two structural facts drive everything:
#   1. NAMING — optimum exported fused names like
#      model.layers.3.attn.q_proj.MatMul.weight, while transformers v5 stores
#      model.language_model.layers.3.self_attn.q_proj.weight. graph_to_hf() is the
#      exact family map (attn→self_attn, gdn→linear_attn, fused-infix strip,
#      tied lm_head→embed_tokens, layer-24 final norm→model.language_model.norm,
#      q/k_norm.layernorm→q/k_norm).
#   2. ORIENTATION — the exporter folded the Linear transpose: graph weight dims
#      are the REVERSE of the HF (out,in) shapes (graph [1024,4096] vs HF
#      (4096,1024)). The only correct transform is a true .T — a same-size reshape
#      would silently scramble every weight.
# LoRA touches only the decoder's Linear projections, but mapping + transplanting
# EVERYTHING that maps is still correct: untouched tensors write identical values.
# The reference's embed_tokens*/vision_encoder* graph FILES stay untouched for the
# same reason (their weights are byte-identical to base).

def graph_to_hf(init_name: str):
    """Exact structural mapping from the reference graph's initializer names to
    the transformers-v5 state dict (verified offline; see the block comment)."""
    n = init_name
    if n.startswith("/model/constants/"):   # exporter-baked constants (shapes, eps, limits)
        return None
    if n == "model.inv_freq":                # RoPE buffer — config-derived
        return None
    if n.endswith("A_neg_exp"):              # derived: exp(-A_log); A_log is not LoRA-targeted
        return None
    n = re.sub(r"\.(MatMul|Gemm|Transpose)\.weight$", ".weight", n)  # optimum fused-op infix
    n = n.replace("conv1d.weight_3d", "conv1d.weight")  # exporter's 3-D view of the same conv weight
    if n == "lm_head.weight":                # tied to the embedding in Qwen3.5
        return "model.language_model.embed_tokens.weight"
    if n.startswith("model.layers."):
        if n.endswith("final_norm_layernorm.weight"):   # graph names it layer 24; HF: model norm
            return "model.language_model.norm.weight"
        n = "model.language_model.layers." + n[len("model.layers."):]
        n = n.replace(".attn.", ".self_attn.")
        n = n.replace(".gdn.", ".linear_attn.")
        n = n.replace("_norm.layernorm.weight", "_norm.weight")
    return n

def find_weight(init_name: str):
    """Initializer name → state-dict key. Exact structural map first, then raw
    name / drifted variants, then a UNIQUE-suffix fallback. Ambiguous suffix
    matches return None — never guess a weight."""
    hf = graph_to_hf(init_name)
    if hf is None:
        return None
    candidates = [hf, init_name,
                  init_name.removeprefix("module."),
                  init_name.split("onnx::")[-1],
                  init_name.replace("onnx::", "").replace("~", ".")]
    for cand in candidates:
        if cand in state:
            return cand
    matches = {sk for sk in state
               if len(init_name) >= 8 and (init_name.endswith(sk) or sk.endswith(init_name))}
    return next(iter(matches)) if len(matches) == 1 else None

# non-persistent / derived buffers — baked from config or recomputed by the
# exporter; they must never be transplanted (and never count as missing)
NO_TRANSPLANT_INITS = re.compile(r"(inv_freq|rotary_emb|position_ids|cached_|A_neg_exp)", re.I)

TRANSPLANT_FILES = ["onnx/decoder_model_merged.onnx", "onnx/decoder_model_merged_fp16.onnx"]
MIN_COVERAGE = 0.70
report = {}
for rel in TRANSPLANT_FILES:
    onnx_path = work / rel
    m = onnx.load(str(onnx_path), load_external_data=True)
    matched, missing, mismatched, ambiguous = 0, [], [], []
    transposed, squeezed, skipped = 0, 0, []
    total = len(m.graph.initializer)
    for init in m.graph.initializer:
        if init.name.startswith("/model/constants/") or NO_TRANSPLANT_INITS.search(init.name):
            skipped.append(init.name)
            continue
        sk = find_weight(init.name)
        if sk is None:
            missing.append(init.name)
            continue
        arr = state[sk].detach().cpu().numpy()
        dims = list(init.dims)
        if dims and list(arr.shape) != dims:
            if arr.ndim == 2 and dims == list(arr.shape)[::-1]:
                arr = arr.T.copy()        # exporter folded the Linear transpose — true .T
                transposed += 1
            elif arr.ndim == 3 and arr.shape[1] == 1 and dims == [arr.shape[0], arr.shape[2]]:
                arr = arr.reshape(dims)   # conv1d: HF (C,1,K) → graph (C,K) — lossless axis-1 squeeze
                squeezed += 1
            elif arr.size == int(np.prod(dims)):
                ambiguous.append((init.name, list(arr.shape), dims))
                continue
            else:
                mismatched.append(init.name)
                continue
        # keep the graph's declared dtype (fp16 graphs stay fp16)
        if init.data_type == TensorProto.FLOAT16:
            arr = arr.astype(np.float16)
        elif init.data_type == TensorProto.FLOAT:
            arr = arr.astype(np.float32)
        init.CopyFrom(numpy_helper.from_array(arr, name=init.name))
        matched += 1
    effective = total - len(skipped)
    coverage = matched / max(1, effective)
    report[rel] = {
        "initializers": total, "matched": matched, "transposed": transposed,
        "squeezed": squeezed, "missing": len(missing),
        "shape_mismatch": len(mismatched), "size_ambiguous": len(ambiguous),
        "buffers_skipped": len(skipped), "coverage": round(coverage, 3),
    }
    if missing:
        print(f"  missing in {rel}: {missing[:12]}{' …' if len(missing) > 12 else ''}")
    assert coverage >= MIN_COVERAGE, (
        f"{rel}: transplant coverage {coverage:.0%} < {MIN_COVERAGE:.0%} — inspect the "
        "report and extend graph_to_hf() before shipping."
    )
    assert not ambiguous, (
        f"{rel}: size-equal shape mismatches {ambiguous[:6]} — a blind reshape would "
        "silently scramble transposed weights; extend graph_to_hf() instead."
    )
    assert not mismatched, f"{rel}: true shape mismatches {mismatched[:6]}"
    onnx.save_model(
        m, str(onnx_path),
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=onnx_path.name + ".data",   # relative to the model file's own dir
        size_threshold=1024,
    )

print(json.dumps(report, indent=2))

# 5.2b — regenerate the QUANTIZED decoder variants from the transplanted graphs.
# The reference's _q4/_q4f16/_quantized files carry BASE weights (int4/int8-packed
# initializers a float transplant can never reach): shipping them untouched would
# silently serve the BASE model to every WebGPU user selecting dtype "q4".
# ORT >=1.20 renamed the module AND the class (MatMul4BitsQuantizer ->
# MatMulNBitsQuantizer, new dep: onnx_ir); older ORT keeps matmul_4bits_quantizer.
try:
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer as NBitsQuantizer
except ModuleNotFoundError:
    try:
        from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer as NBitsQuantizer
    except ModuleNotFoundError as e:
        raise SystemExit('onnxruntime n-bits quantizer unavailable — `pip install onnx_ir onnxruntime` and re-run this cell') from e

def quantize_nbits(src: Path, dst: Path):
    m = onnx.load(str(src), load_external_data=True)
    q = NBitsQuantizer(m, block_size=32, is_symmetric=True,
                       op_types_to_quantize=["MatMul", "Gemm"])
    q.process()
    proto = q.model.model if hasattr(q.model, "model") else q.model
    onnx.save_model(proto, str(dst),
                    save_as_external_data=True, all_tensors_to_one_file=True,
                    location=dst.name + ".data", size_threshold=1024)

quantize_nbits(work / "onnx/decoder_model_merged.onnx",
               work / "onnx/decoder_model_merged_q4.onnx")
print("regenerated q4 decoder from transplanted fp32 graph")
try:
    quantize_nbits(work / "onnx/decoder_model_merged_fp16.onnx",
                   work / "onnx/decoder_model_merged_q4f16.onnx")
    print("regenerated q4f16 decoder from transplanted fp16 graph")
except Exception as e:
    # loud fallback: delete the variant instead of shipping stale base weights
    print("! q4f16 regeneration failed — deleting the variant:", e)
    for p in work.glob("onnx/decoder_model_merged_q4f16*"):
        p.unlink()

# stale int8 variant: not regenerable with ORT dynamic quantization one-to-one,
# and unused by the documented worker contract (dtype q4) — remove, don't ship base.
for p in sorted(work.glob("onnx/decoder_model_merged_quantized*")):
    p.unlink()
    print("removed stale base-weight int8 variant:", p.name)

# superseded external-data blobs (ref naming: .onnx_data / .onnx_data_1)
for name in ["decoder_model_merged", "decoder_model_merged_fp16",
             "decoder_model_merged_q4", "decoder_model_merged_q4f16",
             "decoder_model_merged_quantized"]:
    for p in work.glob(f"onnx/{name}.onnx_data*"):
        p.unlink()

# 5.3 — tokenizer + configs from the merged model (chat_template included)
TOKEN_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
               "generation_config.json", "chat_template.jinja", "added_tokens.json"]
for name in TOKEN_FILES:
    src = Path(MERGED_DIR) / name
    if src.exists():
        shutil.copy(src, work / name)
# older transformers keep the template inside tokenizer_config.json
tc_path = work / "tokenizer_config.json"
if tc_path.exists() and not (work / "chat_template.jinja").exists():
    cfg_json = json.loads(tc_path.read_text())
    if "chat_template" in cfg_json:
        (work / "chat_template.jinja").write_text(cfg_json["chat_template"])
print("tokenizer + configs staged")

# 5.4 — onnxruntime shape verification: every decoder graph we SHIP must load and
# expose the documented IO (float transplants + regenerated q4/q4f16).
import onnxruntime as ort
print("ORT providers:", ort.get_available_providers())
main_models = sorted(work.glob("onnx/decoder_model_merged*.onnx"))
assert main_models, "no decoder graphs left to verify — regeneration failed?"
for p in main_models:
    sess = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    ins = [(i.name, i.shape) for i in sess.get_inputs()]
    outs = [(o.name, o.shape) for o in sess.get_outputs()]
    print(f"✓ {p.relative_to(work)}  inputs={ins[:3]}… outputs={outs[:2]}…")

# 5.5 — upload (private; flip public after inspection)
api.create_repo(ONNX_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=str(work), repo_id=ONNX_REPO, repo_type="model",
                  commit_message="Shadow-1-Whisper: transplanted language weights + tokenizer onto WebGPU ONNX graph")
print(f"✅ ONNX package → https://huggingface.co/{ONNX_REPO}")

print("""
NEXT — flip the three repos PUBLIC, then wire the web runtime:
  Qwen3_5ForConditionalGeneration + AutoProcessor from
  "shadowtalkai/Shadow-1-Whisper-0.8B-ONNX", dtype q4, device webgpu,
  temperature 0.7 / top_p 0.8 / top_k 20 / presence_penalty 1.5
(see README.md — WebGPU worker wiring section)
""")
